# *preparação de datasets*

In [1]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

LABEL_MAP = {
    'gpt': 'openai', 'chatgpt': 'openai', 'openai': 'openai',
    'llama': 'meta', 'meta': 'meta',
    'gemini': 'google', 'gemma': 'google', 'palm': 'google', 'google': 'google',
    'claude': 'anthropic', 'anthropic': 'anthropic',
    'human': 'human', 'wiki': 'human', 'reddit': 'human', 'news': 'human',
}

def map_label(raw):
    raw = str(raw).lower()
    for key,cls in LABEL_MAP.items():
        if key in raw:
            return cls
    return None

def truncate_text(text, max_chars=150):
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars]
    last_space = truncated.rfind(' ')
    return truncated[:last_space] if last_space > 0 else truncated

C:\Users\ricar\miniconda3\envs\condaDAA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## *dataset de exemplo*

In [2]:
df_stor = pd.read_csv("../datasets/dataset-exemplos.csv", sep=";")

df_stor = df_stor.rename(columns={'Text': 'text', 'Label': 'label'})

df_stor['label'] = df_stor['label'].apply(map_label)

df_stor = df_stor.dropna(subset=['label'])

print('dataset de exemplo:', df_stor.shape)
print(df_stor['label'].value_counts())

dataset de exemplo: (125, 3)
label
human        52
anthropic    23
meta         17
openai       17
google       16
Name: count, dtype: int64


## *OpenTuringBench*

In [3]:
print("A transferir OpenTuringBench...")
dataset_otb = load_dataset("MLNTeam-Unical/OpenTuringBench", "in_domain")

dfs_otb = [dataset_otb[split].to_pandas() for split in dataset_otb.keys()]
df_otb = pd.concat(dfs_otb, ignore_index=True)

print("Colunas originais encontradas:", df_otb.columns.tolist())

text_col = next((c for c in df_otb.columns if c.lower() in ['text', 'content', 'generation']), df_otb.columns[0])
label_col = next((c for c in df_otb.columns if c.lower() in ['model', 'label', 'source', 'generator', 'author']), df_otb.columns[1])

df_otb = df_otb.rename(columns={text_col: 'text', label_col: 'label'})

df_otb['label'] = df_otb['label'].apply(map_label)

df_otb = df_otb.dropna(subset=['label'])

print(f"OpenTuringBench processado: {df_otb.shape}")
print(df_otb['label'].value_counts())

A transferir OpenTuringBench...


Colunas originais encontradas: ['url', 'content', 'model']
OpenTuringBench processado: (82852, 3)
label
meta      41426
google    41426
Name: count, dtype: int64


## *HC3*

In [4]:
df_hc3 = pd.read_json("https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl", lines=True)
print('columns:', df_hc3.columns.tolist())
print('total:', len(df_hc3))

columns: ['question', 'human_answers', 'chatgpt_answers', 'index', 'source']
total: 24322


In [5]:
rows = []
for _,item in df_hc3.iterrows():
    for text in (item.get('human_answers') or []):
        if text and len(text.strip()) >= 50:
            rows.append({'text': text.strip(), 'label': 'human'})
    for text in (item.get('chatgpt_answers') or []):
        if text and len(text.strip()) >= 50:
            rows.append({'text': text.strip(), 'label': 'openai'})

df_hc3 = pd.DataFrame(rows)
print('HC3:', df_hc3.shape)
print(df_hc3['label'].value_counts())

HC3: (84480, 2)
label
human     57638
openai    26842
Name: count, dtype: int64


### juntar, truncar e balancear

In [6]:
df_all = pd.concat([df_hc3, df_otb, df_stor], ignore_index=True)

df_all = df_all.dropna(subset=['text', 'label']).drop_duplicates(subset='text')

print('Total antes de balancear:', df_all.shape)
print(df_all['label'].value_counts())

Total antes de balancear: (161494, 4)
label
human        52348
meta         41443
google       41442
openai       26238
anthropic       23
Name: count, dtype: int64


In [7]:
df_all['text'] = df_all['text'].apply(truncate_text)
df_all = df_all[df_all['text'].str.len() >= 50].reset_index(drop=True)
print('Após truncar:', df_all.shape)
print(df_all['text'].str.len().describe().round(1))

Após truncar: (161494, 4)
count    161494.0
mean        143.8
std          11.6
min          50.0
25%         144.0
50%         146.0
75%         148.0
max         150.0
Name: text, dtype: float64


In [8]:
MAX_PER_CLASS = 5000

def balance_class(x):
    if len(x) >= MAX_PER_CLASS:
        return x.sample(n=MAX_PER_CLASS, random_state=42)
    else:
        return x.sample(n=MAX_PER_CLASS, replace=True, random_state=42)

df_balanced = (
    df_all.groupby('label', group_keys=False)
    .apply(balance_class)
    .sample(frac=1, random_state=42)
)

print('Após balanceamento:', df_balanced.shape)
print(df_balanced['label'].value_counts())

Após balanceamento: (25000, 4)
label
google       5000
openai       5000
human        5000
anthropic    5000
meta         5000
Name: count, dtype: int64


C:\Users\ricar\AppData\Local\Temp\ipykernel_13848\4145527635.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(balance_class)


In [9]:
df_train, df_temp = train_test_split(df_balanced, test_size=0.3,
                                     stratify=df_balanced['label'], random_state=42)
df_val, df_test   = train_test_split(df_temp, test_size=0.5,
                                     stratify=df_temp['label'], random_state=42)

print(f'treino: {len(df_train)}  validação: {len(df_val)}  teste: {len(df_test)}')

def save_format(df, prefix, path):
    df = df[['text', 'label']].reset_index(drop=True).copy()
    
    df.insert(0, 'ID', [f'{prefix}-{i+1}' for i in range(len(df))])
    df.columns = ['ID', 'Text', 'Label']
    df['Label'] = df['Label'].str.capitalize()
    df.to_csv(path, sep=';', index=False)
    print(f'guardado em: {path}')

save_format(df_train, 'TR', '../datasets/dataset_train.csv')
save_format(df_val,   'VL', '../datasets/dataset_val.csv')
save_format(df_test,  'TE', '../datasets/dataset_test.csv')

treino: 17500  validação: 3750  teste: 3750
saved at: ../datasets/dataset_train.csv
saved at: ../datasets/dataset_val.csv
saved at: ../datasets/dataset_test.csv
